# FHIRPathを試してみよう！

## 📓学習の目的

[FHIRPath](https://hl7.org/fhirpath/?utm_source=chatgpt.com) を利用して FHIR リソース内の情報を辿り、患者情報や検査結果などの FHIR データ構造を理解できるようになる。

さらに、医療データ標準化やデータ利活用の基礎理解へ繋げる。


## 事前準備

VSCodeの以下エクステンションのインストールが必要です。

- Jupyter
- Python
- REST Client

## 演習1

以下、Pythonパッケージをインストールします。
- FHIRPath を Python で実行するために必要なパッケージ(fhirpathpy)
- FHIRリポジトリへ REST でアクセスするために使用するパッケージ(requests)

**💡セルを実行するには Shift+Enter をクリックします**

In [ ]:
!pip install fhirpathpy
!pip install requests

## 演習2

fhirpathpy の evaluate を使用して、FHIR リソースの中から FHIRPath を使用して値を取得してみよう！

💡 使用例
```python
result = evaluate(patient, "FHIRPath式", [])
print(result)
```

In [ ]:
from fhirpathpy import evaluate

### その1：患者データを探る

変数 patient にサンプルの Patient リソースを設定します（Pythonではディクショナリとして設定されます）。

In [ ]:
patient={
    "resourceType": "Patient",
    "meta": {
        "profile": [
        "http://jpfhir.jp/fhir/core/StructureDefinition/JP_Patient"
        ]
    },
    "identifier": [
        {
        "system": "http://example.org/fhir/sid/tokutei-kenshin-anonymous-id",
        "value": "TK-ANON-0001"
        }
    ],
    "active": True,
    "name": [
        {
        "extension": [
            {
            "url": "http://hl7.org/fhir/StructureDefinition/iso21090-EN-representation",
            "valueCode": "IDE"
            }
        ],
        "use": "usual",
        "text": "佐藤 花子",
        "family": "佐藤",
        "given": [
            "花子"
        ]
        },
        {
        "extension": [
            {
            "url": "http://hl7.org/fhir/StructureDefinition/iso21090-EN-representation",
            "valueCode": "SYL"
            }
        ],
        "use": "usual",
        "text": "サトウ ハナコ",
        "family": "サトウ",
        "given": [
            "ハナコ"
        ]
        },
        {
        "extension": [
            {
            "url": "http://hl7.org/fhir/StructureDefinition/iso21090-EN-representation",
            "valueCode": "ABC"
            }
        ],
        "use": "usual",
        "text": "SATO HANAKO",
        "family": "SATO",
        "given": [
            "HANAKO"
        ]
        }
    ],
    "gender": "female",
    "birthDate": "1978-04-15",
    "address": [
        {
        "use": "home",
        "type": "physical",
        "text": "東京都文京区",
        "city": "文京区",
        "state": "東京都",
        "country": "JP"
        }
    ]
}

#### Q.患者の名前は？

In [ ]:
result = evaluate(patient,"Patient.name", [])
print(result)

#### Q.登録されている患者の漢字氏名？（漢字氏名全体の取得でOK）

💡　ヒント：where(FHIRPath式="値") で絞込

In [ ]:
result = evaluate(patient,"Patient.name.where(extension.valueCode='IDE').given", [])
print(result)

#### Q.登録されている患者のカナ氏名のFirstNameは？

In [ ]:
result = evaluate(patient,"Patient.name.where(extension.valueCode='SYL').given", [])
print(result)

#### Q.登録されている患者の姓名をすべてを取得してみよう

💡ヒント：FHIRPath式の | は結合

In [ ]:
result= evaluate(patient,"Patient.name.where(use='usual').given | name.where(use='usual').family", [])
print(result)

#### Q.登録されている患者の姓名の「最初」の情報を取得してみよう

In [ ]:
result= evaluate(patient,"Patient.name.where(use='usual').given.first() | name.where(use='usual').family.first()", [])
print(result)

#### Q.登録されている患者の姓名の最後に来る文字列を返す

In [ ]:
result= evaluate(patient,"name.where(use='usual').given.last() | name.where(use='usual').family.last()", [])
print(result)

#### Q.患者の誕生日は？

In [ ]:
result = evaluate(patient,"Patient.birthDate")
print(result)

### その2：検査結果（Observation）を探る

変数 obs にサンプルの Observation リソースのディクショナリを設定します。

In [ ]:
obs={
    "resourceType": "Observation",
    "meta": {
        "profile": [
        "http://jpfhir.jp/fhir/core/StructureDefinition/JP_Observation_LabResult"
        ]
    },
    "status": "final",
    "category": [
        {
        "coding": [
            {
            "system": "http://terminology.hl7.org/CodeSystem/observation-category",
            "code": "laboratory",
            "display": "Laboratory"
            }
        ],
        "text": "Laboratory"
        }
    ],
    "code": {
        "coding": [
        {
            "system": "http://loinc.org",
            "code": "2571-8",
            "display": "Triglyceride [Mass/volume] in Serum or Plasma"
        }
        ],
        "text": "中性脂肪"
    },
    "subject": {
        "reference": "urn:uuid:8fa106b8-ecb2-4eea-bb2e-63a5ac5535cc",
        "display": "佐藤 花子"
    },
    "effectiveDateTime": "2025-06-12",
    "valueQuantity": {
        "value": 118,
        "unit": "mg/dL",
        "system": "http://unitsofmeasure.org",
        "code": "mg/dL"
    },
    "note": [
        {
        "text": "2025年度 特定健診サンプルデータ。教材用の架空データです。"
        }
    ]
}

#### Q.この患者さんは、何の検査をしましたか？検査名（textおよびdisplay）とコード(code)、検査結果の数値を調べてみましょう。

In [ ]:
result = evaluate(obs, "code.text")
print(result)

In [ ]:
result = evaluate(obs, "code.coding.display")
print(result)

In [ ]:
result = evaluate(obs, "valueQuantity.value")
print(result)

#### Q.検査結果のコメント（note.text）には何が書かれていますか？

In [ ]:
result = evaluate(obs, "note.text",[])
print(result)

### その3：Bundleリソースから情報を取得してみよう！

In [ ]:
bundle={
    "resourceType": "Bundle",
    "type": "transaction",
    "entry": [
        {
            "resource": {
                "resourceType": "Patient",
                    "address": [
                        {
                        "postalCode": "1600023",
                        "text": "東京都新宿区西新宿6丁目"
                        }
                    ],
                    "birthDate": "1970-01-01",
                    "gender": "male",
                    "identifier": [
                        {
                        "value": "1001"
                        }
                    ],
                    "name": [
                        {
                        "extension": [
                            {
                            "url": "http://hl7.org/fhir/StructureDefinition/iso21090-EN-representation",
                            "valueCode": "IDE"
                            }
                        ],
                        "use": "official",
                        "text": "山田 太郎",
                        "family": "山田",
                        "given": [
                            "太郎"
                        ]
                        },
                        {
                        "extension": [
                            {
                            "url": "http://hl7.org/fhir/StructureDefinition/iso21090-EN-representation",
                            "valueCode": "SYL"
                            }
                        ],
                        "use": "official",
                        "text": "ヤマダ タロウ",
                        "family": "ヤマダ",
                        "given": [
                            "タロウ"
                        ]
                        }
                    ],
                "telecom": [
                    {
                    "system" : "phone",
                    "value" : "0353216200",
                    "use" : "home"
                    }
                ],
                "id": "1"
            },
            "request": {
                "method": "PUT",
                "url": "Patient/1"
            }
        },
        {
            "resource": {
                "resourceType": "Observation",
                "meta": {
                    "profile": [
                        "http://hl7.org/fhir/StructureDefinition/vitalsigns"
                    ]
                },
                "text": {
                    "status": "generated",
                    "div": "<div xmlns=\"http://www.w3.org/1999/xhtml\"><p><b>Generated Narrative with Details</b></p><p><b>id</b>: satO2</p><p><b>meta</b>: </p><p><b>identifier</b>: o1223435-10</p><p><b>partOf</b>: <a>Procedure/ob</a></p><p><b>status</b>: final</p><p><b>category</b>: Vital Signs <span>(Details : {http://terminology.hl7.org/CodeSystem/observation-category code 'vital-signs' = 'Vital Signs', given as 'Vital Signs'})</span></p><p><b>code</b>: Oxygen saturation in Arterial blood <span>(Details : {LOINC code '2708-6' = 'Oxygen saturation in Arterial blood', given as 'Oxygen saturation in Arterial blood'}; {LOINC code '59408-5' = 'Oxygen saturation in Arterial blood by Pulse oximetry', given as 'Oxygen saturation in Arterial blood by Pulse oximetry'}; {urn:iso:std:iso:11073:10101 code '150456' = '150456', given as 'MDC_PULS_OXIM_SAT_O2'})</span></p><p><b>subject</b>: <a>Patient/example</a></p><p><b>effective</b>: 05/12/2014 9:30:10 AM</p><p><b>value</b>: 95 %<span> (Details: UCUM code % = '%')</span></p><p><b>interpretation</b>: Normal (applies to non-numeric results) <span>(Details : {http://terminology.hl7.org/CodeSystem/v3-ObservationInterpretation code 'N' = 'Normal', given as 'Normal'})</span></p><p><b>device</b>: <a>DeviceMetric/example</a></p><h3>ReferenceRanges</h3><table><tr><td>-</td><td><b>Low</b></td><td><b>High</b></td></tr><tr><td>*</td><td>90 %<span> (Details: UCUM code % = '%')</span></td><td>99 %<span> (Details: UCUM code % = '%')</span></td></tr></table></div>"
                },
                "identifier": [
                    {
                        "system": "http://goodcare.org/observation/id",
                        "value": "o1223435-10"
                    }
                ],
                "partOf": [
                    {
                        "reference": "Procedure/ob"
                    }
                ],
                "status": "final",
                "category": [
                    {
                        "coding": [
                            {
                                "system": "http://terminology.hl7.org/CodeSystem/observation-category",
                                "code": "vital-signs",
                                "display": "Vital Signs"
                            }
                        ],
                        "text": "Vital Signs"
                    }
                ],
                "code": {
                    "coding": [
                        {
                            "system": "http://loinc.org",
                            "code": "2708-6",
                            "display": "Oxygen saturation in Arterial blood"
                        },
                        {
                            "system": "http://loinc.org",
                            "code": "59408-5",
                            "display": "Oxygen saturation in Arterial blood by Pulse oximetry"
                        },
                        {
                            "system": "urn:iso:std:iso:11073:10101",
                            "code": "150456",
                            "display": "MDC_PULS_OXIM_SAT_O2"
                        }
                    ]
                },
                "subject": {
                    "reference": "Patient/1"
                },
                "effectiveDateTime": "2021-01-05T09:30:10+01:00",
                "valueQuantity": {
                    "value": 99,
                    "unit": "%",
                    "system": "http://unitsofmeasure.org",
                    "code": "%"
                },
                "interpretation": [
                    {
                        "coding": [
                            {
                                "system": "http://terminology.hl7.org/CodeSystem/v3-ObservationInterpretation",
                                "code": "N",
                                "display": "Normal"
                            }
                        ],
                        "text": "Normal (applies to non-numeric results)"
                    }
                ],
                "device": {
                    "reference": "DeviceMetric/example"
                },
                "referenceRange": [
                    {
                        "low": {
                            "value": 90,
                            "unit": "%",
                            "system": "http://unitsofmeasure.org",
                            "code": "%"
                        },
                        "high": {
                            "value": 99,
                            "unit": "%",
                            "system": "http://unitsofmeasure.org",
                            "code": "%"
                        }
                    }
                ]
            },
            "request": {
                "method": "POST",
                "url": "Observation"
            }
        }
    ]
}


#### Q.Bundle内に何件のリソースが格納されていますか？

💡ヒント：count()を使用します。

In [ ]:
result=evaluate(bundle,"Bundle.entry.count()",[])
print(result)

#### Q.Bundleに含まれるすべてのObservationを抽出します。

In [ ]:
result = evaluate(bundle, "Bundle.entry.resource.where(resourceType='Observation')", [])
print(result)

#### Q.Observationには、何の検査データが格納されていますか？ display と code の値を結合して表示してください。

おまけ：複数の code が設定されている場合は、最後に記載されている情報を入手してみましょう。

In [ ]:
result = evaluate(bundle, "Bundle.entry.resource.where(resourceType='Observation').code.coding.display | Bundle.entry.resource.where(resourceType='Observation').code.coding.code", [])
print(result)
result = evaluate(bundle, "Bundle.entry.resource.where(resourceType='Observation').code.coding.display.last() | Bundle.entry.resource.where(resourceType='Observation').code.coding.code.last()", [])
print(result)

#### Q.検査結果の値はどうなっていますか？

In [ ]:
result = evaluate(bundle, "Bundle.entry.resource.where(resourceType='Observation').valueQuantity.value | Bundle.entry.resource.where(resourceType='Observation')..valueQuantity.unit", [])
print(result)

## その4：FHIRリポジトリから欲しい情報を入手し、FHIRPathで個々の情報を取得してみよう！

問題：佐藤花子さんの全検査データを入手し（検査データは3年分登録されています）BMIの変化を数値で確認した後、グラフで表示してみましょう。

### 1. 花子さんのPatient リソースの ID 入手

FHIRリポジトリから given=花子 で佐藤花子さんリソースを検索し、Patient リソースのIDを入手します。

In [ ]:
import requests
from requests.auth import HTTPBasicAuth
url = "http://localhost/irishealth/csp/healthshare/r4fhirnamespace/fhir/r4/Patient?given=花子"
headers = {
    "Accept": "*/*",
    "content-type": "application/fhir+json",
    "Accept-Encoding": "gzip, deflate, br",
    "Prefer": "return=representation"
}
response = requests.get(url, headers=headers, auth=HTTPBasicAuth('SuperUser', 'SYS'))

In [ ]:
if response.status_code == 200:
    bundle = response.json()
    print("Bundle retrieved successfully!")

    if bundle.get('entry'):
        first_resource = bundle['entry'][0]['resource']
        print("Patient resource ID:",first_resource.get('id'))
else:
    print("Error retrieving bundle:", response.status_code)
    print(response.text)

### 2. 入手した ID を利用して、佐藤花子さんと関連のある全 Observation リソースを入手し、bundle 変数にセットします。

最初に URL を準備をします。ID 番号は入手した番号に変更して実行してください。

💡 ヒント：SearchParameter の patient を利用すると関連する Patient リソースの ID を条件にリソースを入手できます。

In [ ]:
url = "http://localhost/irishealth/csp/healthshare/r4fhirnamespace/fhir/r4/Observation?patient=5"

In [ ]:
response = requests.get(url, headers=headers, auth=HTTPBasicAuth('SuperUser', 'SYS'))
if response.status_code == 200:
    bundle = response.json()
    print("Bundle retrieved successfully!")

    print(f"リソース数：{bundle['total']}")
else:
    print("Error retrieving bundle:", response.status_code)
    print(response.text)

#### ☕ おまけ：Bundle.total と count() の違い

FHIR Search の結果には `Bundle.total` が含まれることがあります。  
これは、検索条件に一致したリソース件数をFHIRサーバが返した値です。

一方、FHIRPath の `count()` は、指定した FHIRPath 式の結果コレクションに含まれる要素数を数える関数です。

そのため、以下の2つは今回たまたま同じ値になります。

- `Bundle.total`
- `Bundle.entry.count()`

しかし、`count()` は任意のパスに対して使えます。

例：

```fhirpath
Bundle.entry.resource.category.coding.where(code='exam').count()

### 3. BMI の coding.code を確認します。

演習で作成した Observation には、BMIの数値を、Observation.code.text に"BMI" として登録しています。FHIRPath を利用して設定時の code と display を入手してください。

In [ ]:
result=evaluate(bundle,"Bundle.entry.resource.code.where(text='BMI').coding.code",[])
print(result)

### 4. coding.code の値を利用して、BMI の過去3年分の値を取得し、変数 bmi に設定します。

In [ ]:
bmi=evaluate(bundle,"Bundle.entry.resource.where(code.coding.code ='39156-5').valueQuantity.value",[])
print(bmi)

### 5.計測した日付（effectiveDateTime）を取得し、変数 effectiveDT に設定します。

In [ ]:
effectiveDT=evaluate(bundle,"Bundle.entry.resource.where(code.coding.code ='39156-5').effectiveDateTime",[])
print(effectiveDT)

### BMI の変遷をグラフ化してみましょう。

💡 ヒント：グラフ化のために、matpltlibパッケージをインストールします。

In [ ]:
!pip install matplotlib

In [ ]:
from decimal import Decimal
import matplotlib.pyplot as plt
# Decimal → float 変換
bmi_float = [float(x) for x in bmi]
plt.plot(effectiveDT,bmi_float, marker="o")

plt.title("BMI Trend")
plt.xlabel("effectiveDateTime")
plt.ylabel("BMI")
plt.grid(True)

plt.show()